# Chart H2 2024's channels by week

*Data Analysis in Python · Topic 8.6*

The channel managers want to explore July to December 2024 themselves: each channel's sessions by week, with the week's revenue when they hover, in a file they can open without Python.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that builds the `sessions` table (lesson 3.'s steps) and the cell that adds its `channel_group` column (topic 4.2.'s steps).
2. In the first cell marked ✏️, create the `weekly_channels` table: one row per Monday-to-Sunday week and channel group, with the columns `date` (the week's Sunday), `channel_group`, `sessions` and `revenue`, and only the weeks that have all 7 days of data.
3. In the second cell marked ✏️, create:
   - `channel_fig`: a Plotly Express line chart of the `weekly_channels` table, with the weeks on the x axis, sessions on the y axis, one line per channel group, the week's revenue in the hover box, and a title
   - `peak_week`: the week with the most sessions, all channels together: the date itself, not text
4. In the last cell marked ✏️, save `channel_fig` to `CHART_FILE` as HTML.
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output. Then download the file from the Files panel, and open it in your browser.

**Expected output**: The `weekly_channels` table has 234 rows, 26 full weeks of 9 channel groups, from the week ending July 7 to the one ending December 29, 2024, with 602,226 sessions. `channel_fig` has nine traces of 26 weeks each, a title, and revenue in the hover box. `peak_week` is the week ending December 1, 2024. `CHART_FILE`, `weekly_channels_h2_2024.html`, has the chart.

## Parameters

In [ ]:
import pandas as pd
import plotly.express as px

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = ["2024-07", "2024-08", "2024-09", "2024-10", "2024-11", "2024-12"]
START, END = "2024-07-01", "2024-12-31"
CHART_FILE = "weekly_channels_h2_2024.html"

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
# Channel groups, with topic 4.2's steps and Foundations topic 12.3's function. Run it, and leave it as it is.
SOCIAL_SITES = {"facebook.com", "m.facebook.com", "l.facebook.com", "lm.facebook.com", "instagram.com", "l.instagram.com"}


def clean(value):
    """Return the value lowercased and stripped, or "" for None."""
    if value is None:
        return ""
    return value.strip().lower()


def channel_group(source, medium):
    """Return the channel group of a session's source and medium, with Foundations topic 12.3's rules."""
    source = clean(source)
    medium = clean(medium)
    if source == "(direct)" and medium == "(none)":
        return "Direct"
    if medium == "email":
        return "Email"
    if medium == "cpc":
        return "Paid Search"
    if medium == "display":
        return "Display"
    if medium == "paid_social":
        return "Paid Social"
    if medium == "video":
        return "Paid Video"
    if medium == "organic":
        return "Organic Search"
    if medium == "referral" and source in SOCIAL_SITES:
        return "Organic Social"
    if medium == "referral":
        return "Referral"
    return "Unassigned"


def row_channel_group(row):
    """Return the channel group of one row of the sessions table."""
    return channel_group(row["session_source"], row["session_medium"])


sessions["channel_group"] = sessions.apply(row_channel_group, axis=1)
sessions["channel_group"].value_counts()

## Weeks by channel

In [ ]:
weekly_channels = sessions.set_index("date").groupby("channel_group")[["sessions", "revenue"]].resample("W").sum().reset_index()
days = sessions.groupby("date")["sessions"].sum().resample("W").count()
full_weeks = days[days == 7].index
weekly_channels = weekly_channels[weekly_channels["date"].isin(full_weeks)]
len(weekly_channels), weekly_channels["date"].nunique()

## The chart

In [ ]:
channel_fig = px.line(weekly_channels, x="date", y="sessions", color="channel_group", hover_data=["revenue"],
                      labels={"date": "Week ending", "sessions": "Sessions", "channel_group": "Channel group", "revenue": "Revenue ($)"},
                      title="Sessions by week and channel group, H2 2024")
channel_fig.update_layout(hovermode="x unified")
channel_fig.show()
peak_week = weekly_channels.groupby("date")["sessions"].sum().idxmax()
peak_week

## The file

In [ ]:
channel_fig.write_html(CHART_FILE)

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables, your chart, your file and the cells that have run. It doesn't change anything.
def check_your_work():
    import os
    import sys
    import pandas as pd
    import plotly.graph_objects as go
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    weekly = found.get("weekly_channels")
    if not isinstance(weekly, pd.DataFrame):
        results.append((False, "weekly_channels should be a DataFrame: sessions.set_index(\"date\").groupby(\"channel_group\")[[\"sessions\", \"revenue\"]].resample(\"W\").sum().reset_index(), with the partial weeks left out."))
    elif not {"date", "channel_group", "sessions", "revenue"} <= set(weekly.columns):
        results.append((False, f"The weekly_channels table should have the columns date, channel_group, sessions and revenue, so Plotly can use them. It has {list(weekly.columns)}: reset_index() turns the labels into columns."))
    elif len(weekly) == 243:
        results.append((False, "The weekly_channels table has 243 rows: 27 weeks of 9 channels, and the last week, ending January 5, 2025, has only December 30 and 31. Keep the weeks with 7 days of data."))
    elif len(weekly) != 234 or weekly["sessions"].sum() != 602226 or weekly["channel_group"].nunique() != 9:
        results.append((False, f"The weekly_channels table should have 234 rows, 26 full weeks of 9 channel groups, with 602,226 sessions. It has {len(weekly)} rows and {weekly['sessions'].sum():,} sessions."))
    else:
        results.append((True, f"234 rows: 26 full weeks of 9 channel groups, from the week ending {weekly['date'].min():%B %-d} to the one ending {weekly['date'].max():%B %-d}, with 602,226 sessions."))

    fig = found.get("channel_fig")
    peak = found.get("peak_week")
    if not isinstance(fig, go.Figure):
        results.append((False, f"channel_fig should be a Plotly figure: px.line(weekly_channels, x=\"date\", y=\"sessions\", color=\"channel_group\", ...). It is a {type(fig).__name__}."))
    else:
        points = [len(trace.y) for trace in fig.data if trace.y is not None]
        if sum(points) == 10974:
            results.append((False, "channel_fig plots 10,974 points: every row of the sessions table, several per channel per day, so the lines zigzag. Plot the weekly_channels table."))
        elif len(points) != 9:
            results.append((False, f"channel_fig should have nine traces, one line per channel group: color=\"channel_group\". It has {len(points)}."))
        elif points != [26] * 9 or abs(sum(sum(trace.y) for trace in fig.data) - 602226) > 0.5:
            results.append((False, f"channel_fig's traces should have 26 weeks each, with weekly_channels' sessions on the y axis. They have {points} points."))
        elif not (fig.layout.title.text or "").strip():
            results.append((False, "channel_fig has no title. Add one: title=\"…\" in px.line()."))
        elif not any("revenue" in (trace.hovertemplate or "").lower() for trace in fig.data):
            results.append((False, "channel_fig's hover box has no revenue. Add it: hover_data=[\"revenue\"] in px.line()."))
        elif not isinstance(peak, pd.Timestamp):
            note = " That's a row number: idxmax() on the long table gives its index. Total by date first." if pd.api.types.is_integer(peak) else ""
            results.append((False, f"peak_week should be the date of the week with the most sessions, all channels together: weekly_channels.groupby(\"date\")[\"sessions\"].sum().idxmax(). It is {peak!r}.{note}"))
        elif peak != pd.Timestamp("2024-12-01"):
            results.append((False, f"peak_week should be the week with the most sessions of all channels together. It is the week ending {peak:%B %-d}."))
        else:
            results.append((True, f"channel_fig has nine channels of 26 weeks, titled {fig.layout.title.text!r}, with revenue in the hover box. The busiest week ended December 1, 2024: Black Friday and the weekend after."))

    chart_file = found.get("CHART_FILE", "weekly_channels_h2_2024.html")
    saved = isinstance(history, list) and any("write_html" in source for source in history)
    if not os.path.exists(chart_file) or not saved:
        results.append((False, f"{chart_file} hasn't been saved in this session. Save the chart: channel_fig.write_html(CHART_FILE)."))
    else:
        with open(chart_file, encoding="utf-8") as file:
            content = file.read()
        if "Organic Search" not in content or "plotly" not in content.lower():
            results.append((False, f"{chart_file} doesn't have the channel chart in it. Save channel_fig: channel_fig.write_html(CHART_FILE)."))
        else:
            results.append((True, f"{chart_file} is saved, {os.path.getsize(chart_file) / 1_000_000:.1f} MB with Plotly inside, so it opens in any browser, without Python."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
weekly_channels = sessions.set_index("date").groupby("channel_group")[["sessions", "revenue"]].resample("W").sum().reset_index()
days = sessions.groupby("date")["sessions"].sum().resample("W").count()
full_weeks = days[days == 7].index
weekly_channels = weekly_channels[weekly_channels["date"].isin(full_weeks)]
len(weekly_channels), weekly_channels["date"].nunique()
```

```python
channel_fig = px.line(weekly_channels, x="date", y="sessions", color="channel_group", hover_data=["revenue"],
                      labels={"date": "Week ending", "sessions": "Sessions", "channel_group": "Channel group", "revenue": "Revenue ($)"},
                      title="Sessions by week and channel group, H2 2024")
channel_fig.update_layout(hovermode="x unified")
channel_fig.show()
peak_week = weekly_channels.groupby("date")["sessions"].sum().idxmax()
peak_week
```

```python
channel_fig.write_html(CHART_FILE)
```

</details>